# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nishu-0618/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


np.random.seed(42)
n_rows = 2500

raw_data = pd.DataFrame({
    'content_id': [f"doc_{i:04d}" for i in range(1, n_rows + 1)],
    'intent': np.random.choice(['informational', 'commercial', 'transactional', 'navigational', None], size=n_rows, p=[0.45, 0.25, 0.20, 0.05, 0.05]),
    'word_count': np.random.choice([np.nan, 350, 800, 1500, 2400, 3800], size=n_rows),
    'days_since_update': np.random.choice([np.nan, 15, 45, 90, 180, 270, 365, 500], size=n_rows),
    'content_age_days': np.random.randint(30, 800, size=n_rows),
    'clicks_m1': np.random.randint(0, 150, size=n_rows),
    'imp_m1': np.random.randint(5, 5000, size=n_rows),
    'pos_m1': np.random.uniform(1.0, 45.0, size=n_rows),
    'clicks_m2': np.random.randint(0, 150, size=n_rows),
    'imp_m2': np.random.randint(5, 5000, size=n_rows)
})

con = duckdb.connect()
df = con.execute("""
    SELECT * FROM raw_data WHERE imp_m1 >= 10
""").df()

# --- 2. Ground Truth Target Construction (Month 2) ---
# Needs refresh: >= 10 baseline clicks and > 20% decline in Month 2 clicks
df['needs_refresh'] = ((df['clicks_m1'] >= 10) & (df['clicks_m2'] < df['clicks_m1'] * 0.80)).astype(int)

# --- 3. Missing Value Imputation / Fills ---
df['word_count'] = df['word_count'].fillna(df['word_count'].median())
df['days_since_update'] = df['days_since_update'].fillna(df['content_age_days'])
df['intent'] = df['intent'].fillna('missing')

# --- 4. Feature Engineering (Month 1 Performance Only) ---
df['ctr_m1'] = df['clicks_m1'] / (df['imp_m1'] + 1e-5)
df['update_to_age_ratio'] = df['days_since_update'] / (df['content_age_days'] + 1.0)
df['is_striking_distance'] = ((df['pos_m1'] >= 11.0) & (df['pos_m1'] <= 20.0)).astype(int)
df['is_page_one'] = (df['pos_m1'] <= 10.0).astype(int)

# --- 5. Categorical Encoding ---
intent_dummies = pd.get_dummies(df['intent'], prefix='intent', drop_first=False, dtype=int)
df = pd.concat([df, intent_dummies], axis=1)

# --- 6. Clean Feature Matrix Assembly ---
feature_cols = [
    'clicks_m1',
    'imp_m1',
    'pos_m1',
    'ctr_m1',
    'word_count',
    'days_since_update',
    'content_age_days',
    'update_to_age_ratio',
    'is_striking_distance',
    'is_page_one'
] + list(intent_dummies.columns)

X = df[feature_cols]
y = df['needs_refresh']

print(f"Feature Vector Shape: {X.shape}")
print(f"Target Distribution:\n{y.value_counts(normalize=True)}")
display(X.head())

Feature Vector Shape: (2499, 15)
Target Distribution:
needs_refresh
0    0.611845
1    0.388155
Name: proportion, dtype: float64


,clicks_m1,imp_m1,pos_m1,ctr_m1,word_count,days_since_update,content_age_days,update_to_age_ratio,is_striking_distance,is_page_one,intent_commercial,intent_informational,intent_missing,intent_navigational,intent_transactional
0,19,2220,7.106139,0.008559,800.0,90.0,386,0.232558,0,1,0,1,0,0,0
1,78,4721,14.201341,0.016522,1500.0,445.0,445,0.997758,1,0,0,0,1,0,0
2,30,432,13.392732,0.069444,2400.0,365.0,544,0.669725,1,0,0,0,0,0,1
3,134,2542,27.586538,0.052714,800.0,90.0,243,0.368852,0,0,1,0,0,0,0
4,57,549,36.936792,0.103825,1500.0,45.0,103,0.432692,0,0,0,1,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Name | Business Meaning | Type / Categorical Handling | Missing Value Strategy | Available Before Prediction? |
| :--- | :--- | :--- | :--- | :--- |
| `clicks_m1` | Total organic search clicks captured in prior month ($T-1$). | Numeric (Continuous) | None (default 0 from warehouse). | **Yes** — Recorded at close of Month 1. |
| `imp_m1` | Total search impressions logged in prior month ($T-1$). | Numeric (Continuous) | Filtered $\ge 10$ to remove unranked noise. | **Yes** — Recorded at close of Month 1. |
| `pos_m1` | Average Google ranking position during Month 1. | Numeric (Continuous) | Replaced with default fallback (100.0) if unranked. | **Yes** — Snapshot from Month 1. |
| `ctr_m1` | Baseline click-through rate (`clicks_m1 / imp_m1`). | Numeric (Engineered ratio) | Added epsilon ($10^{-5}$) to prevent division by zero. | **Yes** — Computed strictly on $T-1$ data. |
| `word_count` | Length of content body text in words. | Numeric (Metadata) | Imputed with dataset median ($1,500$). | **Yes** — Static page attribute at inference. |
| `days_since_update` | Days elapsed since the content was last modified. | Numeric (Metadata) | Imputed with `content_age_days` (assumes unrevised). | **Yes** — Known at inference time. |
| `content_age_days` | Total days elapsed since initial publication. | Numeric (Metadata) | None (mandatory timestamp in CMS). | **Yes** — Known at inference time. |
| `update_to_age_ratio` | Ratio of staleness to total lifespan (`days_since_update / age`). | Numeric (Engineered ratio) | None; denominator bounded by $+1.0$. | **Yes** — Computed from prior metadata. |
| `is_striking_distance` | Indicator flag if ranking position sits in high-ROI zone ($11 \le \text{pos} \le 20$). | Binary (Engineered flag) | None (derived deterministically from `pos_m1`). | **Yes** — Derived strictly from $T-1$ rank. |
| `is_page_one` | Indicator flag if ranking position sits on page 1 ($\text{pos} \le 10$). | Binary (Engineered flag) | None (derived deterministically from `pos_m1`). | **Yes** — Derived strictly from $T-1$ rank. |
| `intent_*` | Query intent classification (`informational`, `commercial`, etc.). | Categorical (One-Hot Encoded) | Imputed with explicit category `'missing'`. | **Yes** — Tagged prior to prediction. |

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score

# 1. Simulate suspicious / leaky features to stress-test the validation pipeline
df['leaky_future_click_drop'] = df['clicks_m2'] - df['clicks_m1']  # Direct label component
df['leaky_post_hoc_flag'] = (df['clicks_m2'] < 5).astype(int)       # Future state proxy

# 2. Compute correlation and Mutual Information with the target (needs_refresh)
test_cols = feature_cols + ['leaky_future_click_drop', 'leaky_post_hoc_flag']
mi_results = []

for col in test_cols:
    # Calculate Mutual Information
    mi = mutual_info_score(df[col].astype(str), y.astype(str))
    # Calculate Linear Correlation where numeric
    corr = df[col].corr(y) if pd.api.types.is_numeric_dtype(df[col]) else np.nan

    # Audit rule: Suspiciously high MI (> 0.25) indicates label leakage or proxy leakage
    status = "🚨 LEAK DETECTED" if mi > 0.25 or abs(corr) > 0.70 else "✅ CLEAN"

    mi_results.append({
        'Feature': col,
        'Mutual Information': round(mi, 4),
        'Correlation with Target': round(corr, 4) if not np.isnan(corr) else 'N/A',
        'Audit Status': status
    })

leakage_audit_df = pd.DataFrame(mi_results).sort_values(by='Mutual Information', ascending=False)
print("=== LEAKAGE STRESS TEST RESULTS ===")
display(leakage_audit_df)

=== LEAKAGE STRESS TEST RESULTS ===


,Feature,Mutual Information,Correlation with Target,Audit Status
2,pos_m1,0.6679,0.0155,🚨 LEAK DETECTED
3,ctr_m1,0.6657,0.0759,🚨 LEAK DETECTED
15,leaky_future_click_drop,0.5830,-0.7946,🚨 LEAK DETECTED
1,imp_m1,0.5155,0.0048,🚨 LEAK DETECTED
7,update_to_age_ratio,0.4892,0.0230,🚨 LEAK DETECTED
6,content_age_days,0.1836,-0.0102,✅ CLEAN
0,clicks_m1,0.1408,0.4412,✅ CLEAN
5,days_since_update,0.0685,0.0143,✅ CLEAN
16,leaky_post_hoc_flag,0.0204,0.1951,✅ CLEAN
4,word_count,0.0011,0.0065,✅ CLEAN


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`clicks_m2` / `imp_m2` / `pos_m2`:** Excluded because these represent performance outcomes during the future evaluation window ($T$), creating direct temporal target leakage.
- **`click_diff` / `pct_click_decay`:** Excluded because computing deltas across the Month 1 $\rightarrow$ Month 2 boundary bakes the target variable directly into the input matrix.
- **`editorial_refresh_ticket_created`:** Excluded because this human operational flag occurs *after* decay is manually spotted, leaking label confirmation.
- **`author_user_id` / `client_internal_id`:** Excluded to eliminate high-cardinality noise, prevent model overfitting on specific domains, and preserve privacy across separate client accounts.
- **`url_slug_raw`:** Excluded in raw format to prevent memorization of specific URL strings in favor of generalizable on-page and historical behavioral features.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.